In [1]:
from probill.agents import TeachableMcpAgent, McpHostAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools
from autogen_agentchat.ui import Console
from typing import Dict
from autogen_ext.experimental.task_centric_memory.utils import ChatCompletionClientRecorder
import json
import os
from probill.utils import create_oai_client, load_yaml_file, create_stdio_server, export_component

config_filepath = "./test.yaml"
config = load_yaml_file(config_filepath)

client = create_oai_client(config["client"])

server_params=create_stdio_server(config["StdioServerParams"])
print(server_params)
mode = "replay" #replay

# client = ChatCompletionClientRecorder(
#     client=client,
#     mode=mode,
#     session_file_path="./sessions/session.json",
# )

command='uvx' args=['mcp-obsidian'] env={'OBSIDIAN_API_KEY': '37d8b00e7e67bd386a2961c37aaf451dc1424ea5d88c37e0c71c7c93eeb04f7c'} cwd=None encoding='utf-8' encoding_error_handler='strict' read_timeout_seconds=5


In [2]:
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write
        ) as session:
            # Initialize the connection
            await session.initialize()
            print("session initiated", flush=True)
            # List available prompts
            # prompts = await session.list_prompts()

            # Get a prompt
            # prompt = await session.get_prompt(
            #     "evaluation", arguments={"patient_id": "P010", "start_page_id": "BINV-20", "clinical_data": {"HER2": "positive"}}
            # )

            # List available resources
            # resources = await session.list_resources()

            # List available tools
            tools = await session.list_tools()
            for tool in tools.tools:
                print(tool.name)
                print(tool.description)
            # # Read a resource
            # content, mime_type = await session.read_resource("file://some/path")

            # Call a tool
            # result = await session.call_tool(
            #     "evaluate_patient_guidelines", 
            #     arguments={
            #         "patient_id": "P010",
            #         "start_page_id": "BINV-20"
            #     }
            # )
            
            # print(result.content)

await run()

session initiated
obsidian_list_files_in_dir
Lists all files and directories that exist in a specific Obsidian directory.
obsidian_list_files_in_vault
Lists all files and directories in the root directory of your Obsidian vault.
obsidian_get_file_contents
Return the content of a single file in your vault.
obsidian_simple_search
Simple search for documents matching a specified text query across all files in the vault. 
            Use this tool when you want to do a simple text search
obsidian_patch_content
Insert content into an existing note relative to a heading, block reference, or frontmatter field.
obsidian_append_content
Append content to a new or existing file in the vault.
obsidian_delete_file
Delete a file or directory from the vault.
obsidian_complex_search
Complex search for documents using a JsonLogic query. 
           Supports standard JsonLogic operators plus 'glob' and 'regexp' for pattern matching. Results must be non-falsy.

           Use this tool when you want to d

In [3]:
agent = McpHostAgent(
    name="obsidian_agent",
    model_client=client,
    description="Agent interacts with Obsidian contents",
    server_params=server_params,
    model_client_stream=True,
    system_message="You are an agent uses MCP tools to fufill the user's request. Say TERMINATE when you done.",
    reflect_on_tool_use=True,
)

In [4]:
task = """List documents"""
result = await Console(agent.run_stream(task=task), output_stats=True)

---------- user ----------
List documents
List documents
---------- obsidian_agent ----------
[FunctionCall(id='call_o7vmgfnk', arguments='{}', name='obsidian_list_files_in_vault')]
[Prompt tokens: 0, Completion tokens: 0]
---------- obsidian_agent ----------
[
  "2024-03-19.md",
  "KDSS sample scenario.md",
  "KDSS system design.md"
]
---------- obsidian_agent ----------
Here are the documents listed in your vault:

1. 2024-03-19.md
2. KDSS sample scenario.md
3. KDSS system design.md

TERMINATE
[Prompt tokens: 0, Completion tokens: 0]
---------- Summary ----------
Number of messages: 4
Finish reason: None
Total prompt tokens: 0
Total completion tokens: 0
Duration: 3.85 seconds


In [6]:
type(client)

autogen_ext.experimental.task_centric_memory.utils.chat_completion_client_recorder.ChatCompletionClientRecorder

In [6]:
agent_json = export_component(agent)
print(agent_json.model_dump_json(indent=2))

{
  "provider": "probill.agents.McpHostAgent",
  "component_type": "agent",
  "version": 1,
  "component_version": 1,
  "description": "Agent interacts with Obsidian contents",
  "label": "obsidian_agent",
  "config": {
    "name": "obsidian_agent",
    "model_client": {
      "provider": "autogen_ext.models.openai.OpenAIChatCompletionClient",
      "component_type": "model",
      "version": 1,
      "component_version": 1,
      "description": "Chat completion client for OpenAI hosted models.",
      "label": "OpenAIChatCompletionClient",
      "config": {
        "frequency_penalty": 0.0,
        "max_tokens": 4096,
        "presence_penalty": 0.0,
        "temperature": 0.8,
        "top_p": 1.0,
        "model": "qwen2.5-coder:32b-instruct-q5_0",
        "api_key": "**********",
        "max_retries": 65535,
        "model_info": {
          "vision": false,
          "function_calling": true,
          "json_output": true,
          "family": "Unknown",
          "structured_outp